In [13]:
from __future__ import annotations

This tells Python to treat  **type annotations as strings**, not as immediately evaluated objects.

### **Why it matters here**

-   You can reference classes  **before they are defined**
-   You avoid circular import issues in larger architecture
-   Your type hints become:
    -   lighter at runtime
    -   easier to refactor
    -   friendlier to tools like  mypy

### **In practice**
This allows things like:
```python
def load(self) -> SchemaSpec:
    ...
```

even if  SchemaSpec  is defined later or in another module
**Rule of thumb**
In any system with interfaces, protocols, and multiple modules → always use this import.


In [12]:
from dataclasses import dataclass

Used to define **simple immutable data containers**, for example:

-   SchemaSpec
-   ValidationEngine

Dataclasses:
-   remove boilerplate (__init__,  __repr__)
-   make data-centric objects explicit
-   fit well with architectural layers

In [ ]:
from typing import Any, Iterable, Iterator, Mapping, Protocol, runtime_checkable

These imports define your contracts and data flow shapes.

In [ ]:
Record = Mapping[str, Any]          # one row from CSV/JSON/etc.

A **single logical row** of data.
Examples:
-   one CSV row
-   one JSON object
-   one database record

 Mapping  is an  **interface**, not a concrete type, it allows:
   -   dict
   -   OrderedDict
   -   read-only mappings

Prevents validators from mutating records
This is a  _contract_, not an implementation.

What if I want to read csv file and read records?

A raw CSV file line is usually:

```
123,John,john@example.com,2024-01-01
```

Or in Python, after reading the file:

```
"123,John,john@example.com,2024-01-01"
```

This is **NOT** a Mapping[str, Any]

-   It has no keys

-   It has no field names

-   Validators cannot work with it

A CSV loader **must convert each row into a Mapping**.



### **Typical correct CSV loader behavior**

```
import csv
from typing import Iterator

class CsvBatchSource:
    def __init__(self, path: str):
        self.path = path

    def read(self) -> Iterator[Record]:
        with open(self.path, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                yield row
```

Now each record looks like:

```
{
  "id": "123",
  "name": "John",
  "email": "john@example.com",
  "created_at": "2024-01-01"
}
```


In [ ]:
Records = Iterable[Record]          # stream of rows

A **sequence of records** that can be:

- a list
- a generator
- a file reader
- a streaming source

### Why Iterable  not list

-   avoids forcing everything into memory
-   allows:
    -   lazy evaluation
    -   large files
    -   streaming pipelines

In [1]:
ValidationReport = Any              # keep abstract at this stage

At this architectural level:

-   you don’t want to commit to:

    -   Pandera error format

    -   Pydantic exceptions

    -   custom error objects


-   different validators will produce different outputs

**Architectural intent**

-   keep **engine and interfaces stable**

-   allow **validators to evolve independently**


For now, flexibility is more valuable than strict typing.

In [2]:
@dataclass(frozen=True)
class SchemaSpec:
    """Neutral, tool-agnostic schema representation produced from YAML."""
    name: str
    version: str | None
    raw: dict[str, Any]             # keep it flexible; later you can strongly type it



**What this class represents**



SchemaSpec  is a simple, immutable data object that holds a schema definition loaded from a YAML file.

It does  **not**  perform validation and is  **not**  tied to any specific library (Pydantic, Pandera, etc.).



Its role is to act as a  **neutral contract**  between configuration (YAML) and execution (validators).

----------

**Why @dataclass(frozen=True)**

-   @dataclass removes boilerplate (__init__, __repr__)

-   frozen=True  makes the object immutable after creation

    → schema configuration cannot be accidentally changed during processing




This is important because schema definitions should be stable once loaded.

----------

**Field explanation**

-   name: str

    Human-readable identifier of the schema

    Used for logging, reporting, and debugging

-   version: str | None

    Optional schema version

    Allows schema evolution without forcing versioning everywhere

-   raw: dict[str, Any]

    The full schema content as loaded from YAML

    Kept flexible on purpose so it can later be compiled into:

    -   a Pydantic model

    -   a Pandera schema

    -   or any other validation strategy

In [3]:
# --- Schema definition layer (YAML -> in-memory schema) ---

@runtime_checkable
class SchemaLoader(Protocol):
    """Loads schema definition from YAML (or other source)."""
    def load(self, schema_ref: str) -> "SchemaSpec": ...


In [15]:

# --- Data loading layer (files -> records/batches) ---

@runtime_checkable
class BatchSource(Protocol):
    """Reads data from files and yields records or batches."""
    def read(self) -> Iterator[Record]: ...
    # optionally: def read_batches(self) -> Iterator[list[Record]]: ...

In [4]:

# --- Validation layer (schema + data -> report) ---

@runtime_checkable
class RecordValidator(Protocol):
    """Validates one record at a time (Pydantic-style)."""
    def validate_record(self, record: Record) -> tuple[bool, Any | None]:
        """
        Returns (is_valid, error_info).
        error_info can be exception, dict, list, etc.
        """
        ...


In [5]:

@runtime_checkable
class BatchValidator(Protocol):
    """Validates a batch/df at once (Pandera-style, or your own rules)."""
    def validate_batch(self, records: list[Record]) -> ValidationReport: ...


In [ ]:
# --- Reporting / output layer ---

@runtime_checkable
class ReportSink(Protocol):
    """Persists or prints validation results."""
    def write(self, report: ValidationReport) -> None: ...


## Parse YAML into a neutral spec (you mostly already have this)

In [6]:
class SchemaLoader(Protocol):
    def load(self, schema_ref: str) -> SchemaSpec: ...

## Compile SchemaSpec into a validation “plan”

In [7]:
class ValidationPlan(Protocol):
    """Tool-agnostic container for whatever the validator needs."""
    ...

In [8]:
class PlanCompiler(Protocol):
    """Turns SchemaSpec (YAML) into something a validator can execute."""
    def compile(self, spec: SchemaSpec) -> ValidationPlan: ...


Examples:

- PydanticPlanCompiler → builds a dynamic Pydantic model + extra rules
- deraPlanCompiler → builds a Pandera DataFrameSchema + checks

## Validators should accept the compiled plan (not raw YAML)

In [9]:
class RecordValidator(Protocol):
    def validate_record(self, plan: ValidationPlan, record: Record) -> tuple[bool, Any | None]: ...

In [10]:
class BatchValidator(Protocol):
    def validate_batch(self, plan: ValidationPlan, records: list[Record]) -> ValidationReport: ...

In [11]:
@dataclass(frozen=True)
class ValidationEngine:
    schema_loader: SchemaLoader
    compiler: PlanCompiler
    source: BatchSource
    sink: ReportSink
    record_validator: RecordValidator | None = None
    batch_validator: BatchValidator | None = None

    def run(self, schema_ref: str) -> None:
        spec = self.schema_loader.load(schema_ref)
        plan = self.compiler.compile(spec)

        if self.record_validator:
            results = []
            for rec in self.source.read():
                ok, err = self.record_validator.validate_record(plan, rec)
                results.append({"ok": ok, "error": err})
            self.sink.write({"schema": spec.name, "mode": "record", "results": results})
            return

        if self.batch_validator:
            batch = list(self.source.read())
            report = self.batch_validator.validate_batch(plan, batch)
            self.sink.write({"schema": spec.name, "mode": "batch", "report": report})
            return

        raise ValueError("Provide either record_validator or batch_validator.")